# Unidad 2: Aprendizaje Automático 
## 📈 Curva de Codo — Número Óptimo de Árboles
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Cluster](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-brett-sayles-4486718.jpg)

[Foto de Brett Sayles](https://www.pexels.com/es-es/foto/cable-de-conexion-cables-alambres-mantenimiento-de-cable-panel-de-parche-4486718/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/15_RandomForest_Tuning_Elbow.ipynb)


## 🎯 ¿Qué vamos a aprender?

Una de las preguntas más frecuentes al usar Random Forest es: **¿cuántos árboles necesito?**
La **Curva de Codo** (*Elbow Curve*) nos da la respuesta visualmente.

Al finalizar, vas a poder:
- ✅ Entender el concepto de **rendimiento decreciente** en Random Forest
- ✅ Generar una **Curva de Codo** para visualizar el impacto de `n_estimators`
- ✅ Usar `GridSearchCV` como herramienta de exploración continua
- ✅ Identificar el **punto de equilibrio** entre rendimiento y costo computacional
- ✅ Tomar decisiones informadas sobre la complejidad del modelo

---

## 🧠 Conceptos Clave

### 📈 ¿Qué es la Curva de Codo?

La **Curva de Codo** (*Elbow Curve*) es una técnica de visualización para encontrar el punto de **rendimiento decreciente**: más allá de un cierto valor, agregar más árboles no mejora significativamente el modelo.

```
Accuracy
  1.0 ┤
 0.97 ┤         ●●●●●●●●●●●●●●●● (plateau)
 0.95 ┤      ●●
 0.93 ┤   ●●
 0.90 ┤●●
       └─────────────────────────
        1  10  20  30  50  100
              n_estimators
         ↑
      "Codo": punto de inflexión
```

### ⚖️ Trade-off: Rendimiento vs Tiempo de Cómputo

- 🌲 **Más árboles** → menor varianza → mejor generalización
- ⏱️ **Más árboles** → más tiempo de entrenamiento y predicción
- 💾 **Más árboles** → más memoria de almacenamiento

La Curva de Codo nos ayuda a encontrar el punto donde el **beneficio se aplana**, permitiéndonos elegir un `n_estimators` económico sin sacrificar rendimiento.

### 🔄 Relación con GridSearchCV

Usamos `GridSearchCV` con `n_estimators` desde 1 hasta 99 para obtener el score validado de cada configuración. El atributo `cv_results_['mean_test_score']` contiene el F1 promedio de los 5 folds para cada valor.

> 📌 **Referencias:**
> - Breiman, L. (2001). [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324). *Machine Learning*, 45, 5–32.
> - Probst, P., & Boulesteix, A. L. (2018). [To Tune or Not to Tune the Number of Trees in Random Forest?](https://www.jmlr.org/papers/v18/17-269.html). *JMLR*.
> - Scikit-learn: [GridSearchCV.cv_results_](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

---

## 📦 Paso 1: Importar las Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar el Dataset

In [ ]:
# 📥 Dataset Breast Cancer Wisconsin
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data['data'], columns=cancer_data['feature_names'])
df['target'] = cancer_data['target']

X = df[cancer_data.feature_names].values
y = df['target'].values

print(f'📐 Dataset: {X.shape[0]} muestras × {X.shape[1]} features')

## 🗺️ Paso 3: Definir el Espacio de Búsqueda

Evaluaremos **todos los valores** de `n_estimators` de 1 a 99, para tener una curva suave y detallada.

> ⚠️ **Advertencia:** Esto entrena `99 × 5 = 495` modelos. En este dataset es rápido, pero con datasets grandes puede tardar varios minutos.

In [ ]:
# 📊 Rango completo: 1 a 99 árboles
n_estimators = list(range(1, 100))
param_grid = {'n_estimators': n_estimators}

print(f'🔢 Valores a evaluar: {len(n_estimators)} (de 1 a 99)')
print(f'🤖 Modelos a entrenar: {len(n_estimators) * 5} (5-Fold CV)')

## 🚀 Paso 4: Ejecutar GridSearchCV

In [ ]:
# 🤖 Modelo y búsqueda
rf = RandomForestClassifier(random_state=135)

gs = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    verbose=0  # Sin verbose para no saturar la salida con 495 modelos
)

print('🚀 Iniciando búsqueda exhaustiva...')
print('⏳ Entrenando 495 modelos...')
gs.fit(X, y)
print('\n✅ ¡Búsqueda completada!')
print(f'🏆 Mejor n_estimators: {gs.best_params_["n_estimators"]}')
print(f'📊 Mejor F1 (CV-5):   {gs.best_score_:.4f}')

## 📈 Paso 5: Graficar la Curva de Codo

Visualizamos el F1 score (validado con 5-Fold CV) en función del número de árboles.

In [ ]:
# 📊 Extraer scores
scores = gs.cv_results_['mean_test_score']
best_n = gs.best_params_['n_estimators']
best_s = gs.best_score_

fig, ax = plt.subplots(figsize=(11, 5))

# Curva principal
ax.plot(n_estimators, scores, color='steelblue', linewidth=1.5, label='F1 Score (CV-5)')

# Sombrear la banda de ±std
stds = gs.cv_results_['std_test_score']
ax.fill_between(n_estimators,
                scores - stds,
                scores + stds,
                alpha=0.2, color='steelblue')

# Marcar el punto óptimo
ax.axvline(x=best_n, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Óptimo: n={best_n}')
ax.scatter([best_n], [best_s], color='tomato', zorder=5, s=80)
ax.annotate(f'  n={best_n}\n  F1={best_s:.3f}',
            xy=(best_n, best_s), xytext=(best_n + 3, best_s - 0.005),
            fontsize=10, color='tomato')

ax.set_xlabel('n_estimators (número de árboles)', fontsize=12)
ax.set_ylabel('F1 Score (media k-Fold)', fontsize=12)
ax.set_title('📈 Curva de Codo — Impacto de n_estimators en Random Forest', fontsize=13)
ax.set_xlim(0, 100)
ax.set_ylim(0.90, 1.0)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔍 Paso 6: Análisis de la Curva

Vamos a analizar en detalle cómo varía el rendimiento en distintos rangos de `n_estimators`.

In [ ]:
# 📊 Análisis por rangos
rangos = [
    ('1–10   (muy pocos árboles)',   slice(0, 10)),
    ('11–30  (rango de codo)',       slice(10, 30)),
    ('31–60  (zona estable)',        slice(30, 60)),
    ('61–99  (árboles adicionales)', slice(60, 99)),
]

scores_arr = np.array(scores)

print('📊 F1 promedio por rango de n_estimators:')
print('-' * 50)
for nombre, sl in rangos:
    media = scores_arr[sl].mean()
    maximo = scores_arr[sl].max()
    print(f'  {nombre}: media={media:.4f}, max={maximo:.4f}')

print('-' * 50)
print(f'\n💡 Punto óptimo encontrado: n_estimators = {best_n}')
print(f'   Mejora con respecto a n=1:  {best_s - scores_arr[0]:+.4f}')
print(f'   Mejora respecto a n=10:     {best_s - scores_arr[9]:+.4f}')
print(f'   Diferencia n=50 vs n=99:   {scores_arr[49] - scores_arr[98]:+.4f}')

## 🏁 Conclusiones

En este notebook aprendimos:

1. 📈 La **Curva de Codo** muestra gráficamente el punto donde agregar más árboles ya no mejora significativamente el modelo.
2. ⚖️ Existe un **trade-off** claro: más árboles = más costo computacional, con beneficios decrecientes.
3. 🎯 El "codo" suele aparecer entre los **10 y 30 árboles** en datasets medianos — más allá, la curva se aplana.
4. 📊 La banda de ±std muestra la **variabilidad** entre folds: una banda estrecha indica un modelo estable.
5. 🚀 Para producción, se recomienda elegir un `n_estimators` en la zona del codo + un margen de seguridad.

### ➡️ Próximo notebook: Feature Importances — ¿Qué variables importan más?

---

## 📚 Referencias

- Breiman, L. (2001). [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324). *Machine Learning*, 45, 5–32.
- Probst, P., & Boulesteix, A. L. (2018). [To Tune or Not to Tune the Number of Trees in Random Forest?](https://www.jmlr.org/papers/v18/17-269.html). *JMLR*, 18, 1–18.
- Scikit-learn: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- Géron, A. (2019). *Hands-On Machine Learning*, Cap. 7. O'Reilly.